# Aula 3 · Lab 2 — Produtores: chaves, partições e garantias

## Objetivo
O **produtor** é o cliente que escreve mensagens no Kafka. Neste lab vamos ver, com números reais:

- como o produtor escolhe a **partição** de cada mensagem e por que a **chave** importa;
- como funciona o envio **assíncrono** e o *callback* de entrega;
- o efeito de `acks`, `linger.ms`, `batch.size` e compressão no **throughput** e na **latência**,
  que é o trade-off visto na Aula 2;
- o que a **idempotência** garante.

## Anatomia de uma mensagem Kafka
| Parte | Descrição |
|---|---|
| **chave** (*key*) | opcional; define a partição e, portanto, a ordem relativa entre mensagens |
| **valor** (*value*) | o conteúdo em si (aqui, JSON). Para o Kafka são só bytes: ele não interpreta o conteúdo |
| **headers** | metadados opcionais em pares chave/valor (origem, versão do esquema, id de rastreio...) |
| **timestamp** | momento da criação, preenchido pelo produtor |
| **partição + offset** | atribuídos na escrita. O *offset* é a posição sequencial da mensagem dentro da partição (0, 1, 2...) e nunca muda |

In [ ]:
import json, time, uuid
from collections import Counter, defaultdict
from confluent_kafka import Producer
from comum.config import BOOTSTRAP

TOPICO = "aula3-pedidos"

## Etapa 1 — Primeira mensagem e o callback de entrega

Criamos o produtor com duas configurações de segurança:

| Parâmetro | Valor | O que faz |
|---|---|---|
| `acks` | `all` | O líder só confirma a escrita depois que **todas as réplicas do ISR** gravaram a mensagem. É a opção mais segura (detalhes na Etapa 4) |
| `enable.idempotence` | `True` | Evita mensagens duplicadas quando o produtor reenvia após uma falha de rede (detalhes na Etapa 5) |

**Como o envio funciona:**
1. `produce()` **não envia nada na hora**: ele só coloca a mensagem num buffer em memória e retorna imediatamente;
2. uma thread interna agrupa as mensagens em **lotes** (*batches*) por partição e envia ao líder de cada uma;
3. quando o broker responde, o resultado fica na fila de eventos do produtor;
4. `poll()` ou `flush()` processam essa fila e chamam o **callback** com o erro (se houver) ou os dados da entrega.

`flush()` espera até que todas as mensagens pendentes sejam entregues ou falhem. Sem ele, o programa
pode terminar com mensagens ainda no buffer, que seriam perdidas.

In [ ]:
produtor = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})

def ao_entregar(erro, msg):
    if erro:
        print("ERRO:", erro)
    else:
        print(f"ok  chave={msg.key().decode()}  partição={msg.partition()}  offset={msg.offset()}")

evento = {"pedido_id": "o-001", "cliente": "ana", "valor": 120.0}
produtor.produce(TOPICO, key="ana", value=json.dumps(evento), callback=ao_entregar)
produtor.flush()

**O que observar:** a linha `ok` só aparece depois do `flush()`. A **partição** foi escolhida pelo produtor
a partir da chave `"ana"`, e o **offset** é a posição que a mensagem ganhou nessa partição. Rode a célula
de novo: a partição se mantém e o offset aumenta em 1.

## Etapa 2 — Mesma chave → mesma partição

Quando a mensagem tem chave, a partição é calculada por:

`partição = hash(chave) % número_de_partições`

Como o hash de uma chave é sempre o mesmo, **todas as mensagens com a mesma chave vão para a mesma partição**.
E como a ordem é garantida dentro de uma partição, as mensagens de uma mesma chave são **lidas na ordem
em que foram escritas**. Essa é a forma de obter ordenação no Kafka: não existe ordem global no tópico,
mas existe ordem **por chave**.

Vamos enviar 60 pedidos de 5 clientes e registrar em que partições cada cliente caiu.

In [ ]:
clientes = ["ana", "bruno", "carla", "diego", "elis"]
destino = defaultdict(set)

def registrar(erro, msg):
    if not erro:
        destino[msg.key().decode()].add(msg.partition())

for i in range(60):
    c = clientes[i % len(clientes)]
    produtor.produce(TOPICO, key=c, value=json.dumps({"pedido_id": f"o-{i:03d}", "cliente": c}), callback=registrar)
produtor.flush()

for c in clientes:
    print(f"{c:6s} -> partições {sorted(destino[c])}")

**Resultado esperado:** cada cliente aparece com **uma única partição**. Dois clientes podem compartilhar
a mesma partição (5 chaves para 6 partições, e o hash não garante distribuição perfeita), mas um cliente nunca se divide.

**Consequência prática:** se uma chave for muito mais frequente que as outras (por exemplo, um cliente
corporativo enorme), a partição dela fica sobrecarregada. Isso se chama **hot partition**, e a escolha da
chave deve equilibrar ordenação e distribuição de carga.

## Etapa 3 — E sem chave?

Sem chave, não há exigência de ordem, e o produtor usa o **sticky partitioner** (padrão desde o Kafka 2.4):
ele enche um lote inteiro numa partição e só então passa para outra. Isso gera lotes maiores, e portanto
mais eficientes, do que alternar a partição a cada mensagem (*round-robin*), que era o comportamento antigo.

Vamos enviar 3 000 mensagens sem chave e contar quantas caíram em cada partição.

In [ ]:
contagem = Counter()
for i in range(3000):
    produtor.produce(TOPICO, value=f"sem-chave-{i}",
                     callback=lambda e, m: contagem.update([m.partition()]) if not e else None)
produtor.flush()
print(sorted(contagem.items()))

**O que observar:** a saída é uma lista de pares `(partição, quantidade)`. As mensagens se espalham pelas
partições, mas **não de forma perfeitamente igual**, porque a distribuição acontece por lotes. Em volumes
maiores, ela tende a se equilibrar.

## Etapa 4 — Latência × throughput: `acks`, `linger.ms`, `batch.size` e compressão

Esta etapa mede, com números reais, o trade-off da Aula 2. A função `medir()` envia 50 000 mensagens com
uma configuração e calcula três métricas:

| Métrica | O que significa |
|---|---|
| `msgs/s` | **Throughput**: mensagens confirmadas por segundo |
| `lat_p50_ms` | **Latência mediana**: metade das mensagens foi confirmada em menos que esse tempo, desde o `produce()` até o callback |
| `lat_p99_ms` | **Latência do percentil 99**: 99% das mensagens foram confirmadas em menos que esse tempo. Mostra o "pior caso típico" e costuma ser o número que importa num sistema real |

Os parâmetros testados:

| Parâmetro | O que controla | Efeito esperado |
|---|---|---|
| `linger.ms` | Quanto tempo o produtor **espera** por mais mensagens antes de enviar um lote. `0` = envia assim que puder | Esperar mais forma lotes maiores: **mais throughput**, mas cada mensagem espera mais, com **mais latência** |
| `batch.size` | Tamanho máximo de um lote por partição, em bytes (padrão ~1 MB no librdkafka) | Lotes maiores permitem mais mensagens por requisição. Um lote cheio é enviado mesmo antes de vencer o `linger.ms` |
| `acks` | Quantas confirmações o produtor exige. `0`: nenhuma (dispara e esquece). `1`: só o líder gravou. `all`: todas as réplicas do ISR gravaram | `all` acrescenta o tempo de replicação entre brokers: **mais latência**, mas a mensagem confirmada sobrevive à queda do líder |
| `compression.type` | Algoritmo para comprimir os lotes (`none`, `gzip`, `snappy`, `lz4`, `zstd`) | Menos bytes na rede e no disco, em troca de CPU. Funciona melhor com lotes grandes e dados repetitivos, como JSON |

A célula testa 5 combinações e leva de 1 a 2 minutos. As mensagens vão para o tópico
**`aula3-benchmark`**, criado automaticamente, para não misturar dados de teste com os pedidos que o Lab 3 vai ler.

**Detalhe de implementação:** se o buffer do produtor encher (`BufferError`), a função chama `poll()` para
liberar espaço e tenta de novo. Isso é **backpressure**: o produtor é obrigado a desacelerar quando o
Kafka não consegue absorver no mesmo ritmo.

In [ ]:
TOPICO_TESTE = "aula3-benchmark"   # tópico separado para não poluir aula3-pedidos

def medir(n=50_000, **cfg):
    p = Producer({"bootstrap.servers": BOOTSTRAP, **cfg})
    latencias = []
    payload = json.dumps({"pedido_id": "o-teste", "cliente": "benchmark", "itens": ["produto"] * 20, "valor": 199.9})
    inicio = time.perf_counter()
    for i in range(n):
        t0 = time.perf_counter()
        while True:
            try:
                p.produce(TOPICO_TESTE, value=payload,
                          callback=lambda e, m, t0=t0: latencias.append(time.perf_counter() - t0))
                break
            except BufferError:
                p.poll(0.05)
        p.poll(0)
    p.flush()
    total = time.perf_counter() - inicio
    latencias.sort()
    return {"msgs/s": round(n / total), "lat_p50_ms": round(latencias[len(latencias)//2] * 1000, 1),
            "lat_p99_ms": round(latencias[int(len(latencias)*.99)] * 1000, 1)}

for cfg in [{"linger.ms": 0, "acks": "1"},
            {"linger.ms": 20, "acks": "1"},
            {"linger.ms": 100, "acks": "1", "batch.size": 262144},
            {"linger.ms": 20, "acks": "all"},
            {"linger.ms": 20, "acks": "all", "compression.type": "lz4"}]:
    print(cfg, medir(**cfg))

**Como ler os resultados.** Compare as linhas em pares:

1. **`linger.ms` 0 → 20 → 100** (com `acks=1`): as latências p50 e p99 sobem aproximadamente no valor do
   `linger.ms`, porque cada mensagem espera o lote se formar. O throughput tende a subir ou se manter.
2. **`acks=1` → `acks=all`** (ambos com `linger.ms=20`): a latência aumenta, porque o líder agora espera os
   seguidores copiarem antes de confirmar.
3. **Com e sem `lz4`**: com compressão, menos bytes trafegam e o throughput costuma melhorar.

> **Atenção ao interpretar:** os 3 brokers e o produtor rodam na **mesma VM**, então a "rede" entre eles
> é quase instantânea e todos disputam a mesma CPU. Os números absolutos são menores e mais instáveis do que
> num cluster real. O que vale observar é a **tendência** entre as configurações. Rodar a célula duas vezes
> pode dar resultados um pouco diferentes.

**Para discutir:** por que `acks=all` custa latência? Em que sistemas vale pagar esse preço (pagamentos,
pedidos) e em quais não vale (métricas de monitoramento, logs de cliques)?

## Etapa 5 — Idempotência

**O problema:** o produtor envia uma mensagem, o broker grava, mas a confirmação se perde na rede.
O produtor acha que falhou e **reenvia** (*retry*), e a mensagem fica duplicada no tópico.

**A solução:** com `enable.idempotence=True`, cada produtor recebe um **id** do broker e numera suas mensagens
por partição (números de sequência). Se chegar uma mensagem com um número de sequência que o broker já gravou,
ele a descarta. Ligar a idempotência também força `acks=all` e limita as requisições em andamento, para preservar a ordem.

**O que isso garante:** *exactly-once* na escrita **de um produtor para uma partição**, durante a vida
daquele produtor. Não cobre:
- o seu programa enviar a mesma mensagem duas vezes, por exemplo após reiniciar (são duas mensagens diferentes para o Kafka);
- o processamento de ponta a ponta (ler → processar → escrever). Para isso existem as **transações** do Kafka
  ou sinks idempotentes, como veremos com o Spark na Aula 4.

## Exercícios
1. Envie 10 mensagens com a chave `"zeca"` e confirme pela Kafka UI (*Topics → aula3-pedidos → Messages*) que todas estão na mesma partição.
2. Acrescente cabeçalhos (`headers=[("origem", b"lab2")]`) às mensagens e encontre-os na Kafka UI.
3. Rode `medir()` com `linger.ms` = 0, 5, 50 e 200 e monte um gráfico (matplotlib) de msgs/s × p99.
4. Pelo terminal, produza manualmente com chave:
   `kafka-console-producer.sh --bootstrap-server kafka1:9092 --topic aula3-pedidos --property parse.key=true --property key.separator=:`
   e digite linhas como `ana:{"valor": 10}`. Confira que caem na mesma partição dos pedidos da Ana da Etapa 2.